# 03 -- Paper Parity Backtest (scaffold)

This notebook is intentionally a scaffold: the composed weekly pipeline (`src.backtest.run_week`) is blocked on unresolved paper decisions, starting with `DECISION_REQUIRED_LOOKBACK_N`. Per the rebuild's source-fidelity mandate, this notebook does **not** populate any guessed defaults to force a number out of the backtest -- it demonstrates the explicit failure and reports the full decision-dependency chain instead.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.backtest import run_week
from src.decisions import PaperDecisionRequiredError

## Attempt the composed weekly pipeline

In [2]:
try:
    run_week(week_id=0, pipeline_state={})
    raise SystemExit('ERROR: pipeline ran without raising -- this should not happen while decisions remain unresolved')
except PaperDecisionRequiredError as exc:
    print('PAPER_REBUILD_BLOCKED_ON_USER_DECISIONS')
    print(f'First blocking decision: {exc.decision_id} (Level {exc.level})')
    print(exc)

PAPER_REBUILD_BLOCKED_ON_USER_DECISIONS
First blocking decision: DECISION_REQUIRED_LOOKBACK_N (Level 1)
[DECISION_REQUIRED_LOOKBACK_N] (Level 1) 'Input tensor history depth N' is not resolved in decisions/paper_decision_register.json. Required before: running the composed weekly pipeline for week 0 -- tensor construction (the first pipeline stage) is blocked; see the individual stage modules for the full set of decisions this composed pipeline depends on downstream.. See docs/paper_source_audit.md and the decision register for source evidence and candidate options.


## Full outstanding decision list (by level)

See `decisions/paper_decision_register.json` for full source evidence and candidate options per item.

In [3]:
register = json.loads((PROJECT_ROOT / 'decisions' / 'paper_decision_register.json').read_text())
df = pd.DataFrame(register)[['decision_id', 'level', 'topic', 'source_classification', 'status']]
df = df.sort_values(['level', 'decision_id'])
for level in [1, 2, 3]:
    print(f'--- Level {level} ({(df.level == level).sum()} items) ---')
    for _, row in df[df.level == level].iterrows():
        print(f"  {row.decision_id:55s} {row.source_classification}")
    print()

--- Level 1 (16 items) ---
  DECISION_REQUIRED_BATCH_SIZE                            MISSING
  DECISION_REQUIRED_CUSTOM_FINANCIAL_LOSS                 MISSING
  DECISION_REQUIRED_DROPOUT_RATE                          MISSING
  DECISION_REQUIRED_EARLY_STOPPING                        MISSING
  DECISION_REQUIRED_FRAMEWORK_SUBSTITUTION                EXPLICIT (paper) / environment-triggered contingency
  DECISION_REQUIRED_HIDDEN_WIDTHS                         MISSING
  DECISION_REQUIRED_INITIAL_EPOCHS                        MISSING
  DECISION_REQUIRED_LEARNING_RATE                         MISSING
  DECISION_REQUIRED_LOOKBACK_N                            MISSING
  DECISION_REQUIRED_NORMALIZATION_SCOPE                   MISSING
  DECISION_REQUIRED_OPTIMIZER                             MISSING
  DECISION_REQUIRED_PRICE_FIELD                           MISSING
  DECISION_REQUIRED_RANDOM_SEED_POLICY                    MISSING
  DECISION_REQUIRED_TARGET_RETURN_INTERVAL                STRONG_INFER

## Reference-only: paper's reported parity results

Transcribed verbatim from Table 3 (p.6) for future comparison once decisions are resolved and a real backtest can run. **Not a tuning target** -- see `docs/paper_source_audit.md` #33.

In [4]:
reported = pd.read_csv(PROJECT_ROOT / 'outputs' / 'paper_reported_yearly_metrics.csv')
reported

,year,cagr_current_pct,cagr_spx_pct,alpha_pct,sharpe,max_drawdown,win_pct,buys_per_week
0,2012,15.13,15.88,-0.75,1.76,-0.110,60.51,3.02
1,2013,47.57,32.43,15.14,1.87,-0.090,63.91,3.31
2,2014,21.44,13.81,7.63,1.09,-0.070,61.90,1.65
3,2015,2.72,1.31,1.41,1.46,-0.090,50.00,2.78
4,2016,54.41,11.93,42.48,1.36,-0.070,63.11,1.98
5,2017,22.48,21.94,0.54,1.61,-0.060,63.08,2.50
6,2018,-3.18,-4.41,1.23,0.94,-0.150,54.55,1.27
7,2019,42.46,31.74,10.72,1.91,-0.050,69.33,2.94
8,2020,25.00,18.38,6.62,0.95,-0.100,61.25,1.57
9,2021,54.38,28.83,25.55,1.38,-0.070,60.66,2.35


## Status

`PAPER_REBUILD_BLOCKED_ON_USER_DECISIONS` -- see `docs/paper_rebuild_status.md` for the full classification rationale and the complete list of Level-1/2/3 decisions blocking an executable, honest paper-parity backtest.